# Projet 2 — Analyse de sentiments par Fine-Tuning de DistilBERT

**Environnement recommandé : Google Colab avec GPU (T4)**

Objectif : fine-tuner DistilBERT sur le dataset IMDB pour classifier des avis en positif/négatif, puis exporter le modèle pour une inference locale sur CPU (i3, 8 Go RAM).


## 1. Installation et imports

In [ ]:
!pip install -q transformers==4.40.0 datasets==2.19.0 evaluate accelerate

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
import evaluate

print("Torch version:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())


## 2. Chargement du dataset IMDB

In [ ]:
dataset = load_dataset("imdb")

# Pour accélérer l'entraînement sur Colab (dataset complet = 50k exemples),
# on peut travailler sur un sous-ensemble représentatif :
train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(2000))

print(train_dataset[0])


## 3. Tokenization

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=256)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## 4. Chargement du modèle pré-entraîné

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


## 5. Définition des métriques d'évaluation

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}


## 6. Configuration et lancement du fine-tuning

In [ ]:
training_args = TrainingArguments(
    output_dir="./resultats_distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


## 7. Évaluation finale et matrice de confusion

In [ ]:
eval_results = trainer.evaluate()
print("Résultats sur le set de test :", eval_results)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

predictions = trainer.predict(test_tokenized)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=["négatif", "positif"]))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["négatif", "positif"], yticklabels=["négatif", "positif"], cmap="Blues")
plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.title("Matrice de confusion")
plt.show()


## 8. Sauvegarde du modèle fine-tuné (pour inference locale)

In [ ]:
SAVE_DIR = "modele_sentiment_distilbert"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!zip -rq modele_sentiment_distilbert.zip modele_sentiment_distilbert
print("Modèle sauvegardé et compressé : modele_sentiment_distilbert.zip")


## 9. (Optionnel) Export ONNX + quantification dynamique

Pour réduire encore la charge CPU/RAM en local, on peut convertir le modèle en ONNX et le quantifier en int8. Cette étape est optionnelle mais recommandée si l'inference locale avec PyTorch te paraît trop lente.

In [ ]:
!pip install -q optimum[onnxruntime]

from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ort_model = ORTModelForSequenceClassification.from_pretrained(SAVE_DIR, export=True)
ort_model.save_pretrained("modele_sentiment_onnx")

quantizer = ORTQuantizer.from_pretrained(ort_model)
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
quantizer.quantize(save_dir="modele_sentiment_onnx_quantized", quantization_config=qconfig)

!zip -rq modele_sentiment_onnx_quantized.zip modele_sentiment_onnx_quantized
print("Version ONNX quantifiée exportée (plus légère pour l'inference locale).")


## 10. Téléchargement vers ton PC

In [ ]:
from google.colab import files
files.download("modele_sentiment_distilbert.zip")
# Si tu as fait l'étape 9 :
# files.download("modele_sentiment_onnx_quantized.zip")
